In [5]:
import ee
import geemap
import numpy as np
from datetime import datetime, timedelta
from scipy import stats
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from statsmodels.tsa.seasonal import STL
import matplotlib.pyplot as plt
from pathlib import Path

import re
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
import rasterio
from rasterio.windows import from_bounds
from scipy.ndimage import uniform_filter
from scipy.stats import describe

# Initialize Earth Engine with service account
try:
    service_account_file = "../../training-253313-c905674c1ca0.json"
    if Path(service_account_file).exists():
        print(f"Authenticating with service account: {service_account_file}")
        credentials = ee.ServiceAccountCredentials(None, service_account_file)
        ee.Initialize(credentials)
        print("✓ Google Earth Engine initialized with service account")
    else:
        raise FileNotFoundError("Service account file not found")
except Exception as e:
    print(f"Service account auth failed: {e}")
    print("Trying user authentication...")
    try:
        ee.Initialize()
        print("✓ Google Earth Engine already initialized")
    except:
        print("Authenticating with user credentials...")
        ee.Authenticate()
        ee.Initialize()
        print("✓ Google Earth Engine authenticated and initialized")

Authenticating with service account: ../../training-253313-c905674c1ca0.json
✓ Google Earth Engine initialized with service account


### Tracking Dam Construction using Remote Sensing

**What is SAR?**


Synthetic Aperture Radar (SAR) is a type of remote sensing technology that uses radar signals, rather than visible light, to observe the Earth's surface from satellites. Unlike conventional optical satellites that rely on sunlight and can be blocked by clouds, rain, or darkness, SAR actively emits its own microwave signals toward the ground and records what bounces back. This makes it an all-weather, day-and-night imaging system capable of penetrating cloud cover entirely.

**What is measured?**


When the radar signal strikes the Earth's surface, a portion of that energy is reflected back to the satellite. The strength of this returned signal is called **backscatter intensity**. It essentially tells us how "reflective" a surface is to radar energy. Different surfaces return the signal with very different strengths. Smooth, flat surfaces such as calm water tend to reflect the radar signal away from the satellite, resulting in low backscatter, appearing dark in SAR imagery. Rough or structured surfaces, such as buildings, dense vegetation, or disturbed soil, scatter energy back strongly, appearing bright. This contrast allows analysts to detect and track changes on the ground, such as land clearing, flooding, or construction activity, by observing shifts in backscatter intensity over time.

**Methodology**


We chose the VV polarization (where both the transmitted and received signal are vertically oriented) because its particularly sensitive to surface roughness, structural features, and changes in ground conditions, characteristics that are highly relevant when monitoring construction activity, ground displacement, or changes in water body extent around a dam site. 

For the analysis, we made some choices in terms of band and orbit selection:

- <b>SAR is a side-looking sensor.</b> If data from both Ascending (South to North) and Descending (North to South) orbits are mixed, the satellite views the target from opposite angles. In steep river valleys, this alternating geometry creates severe "sawtooth" noise in the time series due to changing radar layover and shadow effects. By restricting the data strictly to the Ascending pass, we locked the viewing angle, ensuring that any variation in backscatter intensity was caused by physical changes on the ground rather than changes in the sensor's perspective.

- Vertical transmit and Vertical receive (VV) polarization was selected over VH (Vertical-Horizontal) or cross-polarization. <b>VV is generally more sensitive to surface scattering and vertical geometries</b>, making it the optimal choice for detecting the hard, angular surfaces of dam walls, spillways, and heavy construction equipment.

<br>

Rather than analyzing individual satellite passes (which occur every 6 to 12 days), the time series was aggregated into Monthly Median Composites for two reasons:

- <b>Speckle and Transient Noise Reduction:</b> Raw SAR images contain inherent granular noise known as "speckle." Additionally, active construction sites feature transient high-reflectance targets (e.g., moving cranes, temporary scaffolding, or parked trucks). Taking the median value of all passes within a calendar month effectively filters out these temporary anomalies, leaving only the stable, permanent structural changes.

- <b>Computational Efficiency:</b> Aggregating to monthly medians drastically reduced the data volume, preventing memory timeouts during cloud-based processing while preserving the macro-level timeline of the multi-year construction project.
<br><br>

We're also considering a control point in the analysis (typically an upstream location in the river), to create a functional counter-signal to the dam construction. While the dam point monitors "building up," the reservoir point monitors "filling in," providing a complete picture of the project's operational status. We made the following assumptions:

<b>Detection of First Impoundment (Inverse Correlation)</b>: 

The primary reason to choose an upstream location is to detect the exact moment of First Impoundment (the start of reservoir filling).

- <b>The Inverse Correlation: </b> In a successful project, you expect to see the Dam Wall backscatter stabilize at a high intensity (concrete/steel) while the Reservoir backscatter takes a sudden, permanent dive toward -20 dB or lower.

- <b>Validation of Function:</b> High backscatter at the dam site proves the structure exists, but the drop at the reservoir site proves the structure is actually working to hold back water.


In [295]:
def process_dam_data(dam_coords, res_coords, start_date, end_date, buffer_dam=150, buffer_res=40):
    """
    Analyses Sentinel-1 backscatter trends at dam and reservoir locations over a specified timeline.

    Args:
        dam_coords (list): [longitude, latitude] of the dam location.
        res_coords (list): [longitude, latitude] of the reservoir location.
        start_date (str): Start date in 'YYYY-MM-DD' format.
        end_date (str): End date in 'YYYY-MM-DD' format.
        
    Returns:
        pd.DataFrame: A DataFrame containing mean backscatter values for the dam and reservoir locations
    """
    # Create FeatureCollection
    pts = ee.FeatureCollection([
        ee.Feature(ee.Geometry.Point(dam_coords).buffer(buffer_dam), {'label': 'Dam'}),
        ee.Feature(ee.Geometry.Point(res_coords).buffer(buffer_res), {'label': 'Reservoir'})
    ])
    
    # Filter Sentinel-1 Collection
    s1_col = (ee.ImageCollection('COPERNICUS/S1_GRD')
          .filterBounds(pts)
          .filterDate(start_date, end_date)
          .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))
          .filter(ee.Filter.eq('orbitProperties_pass', 'ASCENDING')) # Choosing only ascending to reduce 'sawtooth' effect
          .filter(ee.Filter.eq('instrumentMode', 'IW')))
    
    # Generate monthly composites with a "Band Check"
    months = ee.List.sequence(0, ee.Date(end_date).difference(ee.Date(start_date), 'month').round().subtract(1))
    
    def create_monthly(n):
        start = ee.Date(start_date).advance(n, 'month')
        end = start.advance(1, 'month')
        subset = s1_col.filterDate(start, end)
        count = subset.size()
        composite = subset.median()
        return composite.set({
            'system:time_start': start.millis(),
            'band_count': composite.bandNames().size()
        })
    
    monthly_col = (ee.ImageCollection.fromImages(months.map(create_monthly))
                   .filter(ee.Filter.gt('band_count', 0)))
    
    def extract_stats(image):
        date = image.date().format('YYYY-MM-dd')
        stats = image.reduceRegions(
            collection=pts,
            reducer=ee.Reducer.mean(),
            scale=10
        )
        return stats.map(lambda f: f.set('date', date))
    
    results = monthly_col.map(extract_stats).flatten().getInfo()
    data = [f['properties'] for f in results['features']]
    df = pd.DataFrame(data)
    df['date'] = pd.to_datetime(df['date'])
    
    return df

In [238]:
def postprocess_dam_data(df, label = 'dam'):
    #df = df.copy()

    if label.lower() == 'dam':
        df = df[df['label'] == 'Dam']
    elif label.lower() == 'reservoir':
        df = df[df['label'] == 'Reservoir']
    else:
        raise ValueError("Label must be either 'dam' or 'reservoir'")

    #df_dam = df[df['label'] == 'Dam']
    #df_res = df[df['label'] == 'Reservoir']

    ## Ensure the index is datetime and has a fixed monthly frequency
    df = df.sort_index()
    df['date'] = pd.to_datetime(df['date'])

    df = df.set_index('date')

    ## Resample the 'VV' values into monthly buckets (taking the mean of each month)
    df_monthly = df['VV'].resample('MS').mean().to_frame()

    ## Interpolating the series since models like STL require a continuous time series without missing values
    df['VV_filled'] = df['VV'].interpolate(method='linear')

    ## Modeling the time series using STL decomposition
    stl = STL(df['VV_filled'], period=12, seasonal=13, robust=True)
    res = stl.fit()

    ## Adding the components back to the dataframe for analysis
    df['trend'] = res.trend
    df['seasonal'] = res.seasonal
    df['deseasonalized'] = df['VV_filled'] - res.seasonal

    ## Check for nulls in the trend component
    print(f"Total NaNs in Trend: {df['trend'].isna().sum()}")   
    
    return df

In [239]:
def plot_trends(df, label):
    fig, (ax1, ax2, ax3, ax4) = plt.subplots(4, 1, figsize=(12, 12), sharex=True)

    ax1.plot(df.index, df['VV'], color='gray', alpha=0.5, label='Original VV')
    ax1.plot(df.index, df['deseasonalized'], color='red', label='Deseasonalized (STL)')
    ax1.set_title(f'Original vs. Deseasonalized {label} Progress')
    ax1.legend()

    ax2.plot(df.index, df['trend'], color='blue')
    ax2.set_title('Trend Component (Long-term Structural Growth)')

    ax3.plot(df.index, df['seasonal'], color='green')
    ax3.set_title('Seasonal Component (Monsoon Cycles)')

    ax4.scatter(df.index, df['VV_filled'] - df['trend'] - df['seasonal'], color='black', s=10)
    ax4.set_title('Residuals (Unexplained Anomalies)')

    plt.tight_layout()
    plt.show()

In [240]:
import altair as alt

def plot_trends_grid(combined_df, dam_ids=None, ncols=3, nrows=2):
    """
    Plot trend components for multiple dams in a grid layout using Altair.
    
    Args:
        combined_df: DataFrame with columns ['dam_id', 'dam_name', 'date', 'trend']
        dam_ids: List of dam_ids to plot. If None, plots first ncols*nrows dams
        ncols: Number of columns in the grid (default: 3)
        nrows: Number of rows in the grid (default: 2)
    
    Returns:
        Altair chart object
    """
    # Select dams to plot
    if dam_ids is None:
        available_dams = combined_df['dam_id'].unique()
        dam_ids = available_dams[:ncols * nrows]
    
    # Filter data for selected dams
    plot_data = combined_df[combined_df['dam_id'].isin(dam_ids)].copy()
    
    # Ensure date is datetime
    plot_data['date'] = pd.to_datetime(plot_data['date'])
    
    # Create base chart
    base = alt.Chart(plot_data).mark_line(
        strokeWidth=2.5,
        color='#0C4A6E'
    ).encode(
        x=alt.X('date:T', 
                title='Date',
                axis=alt.Axis(format='%Y', labelAngle=0)),
        y=alt.Y('trend:Q', 
                title='Backscatter Intensity (dB)',
                scale=alt.Scale(zero=False)),
        tooltip=[
            alt.Tooltip('dam_name:N', title='Dam'),
            alt.Tooltip('date:T', title='Date', format='%Y-%m-%d'),
            alt.Tooltip('trend:Q', title='Trend (dB)', format='.2f')
        ]
    ).properties(
        width=300,
        height=200
    )
    
    # Create faceted chart
    chart = base.facet(
        facet=alt.Facet('dam_name:N', title=None),
        columns=ncols
    ).properties(
        title=alt.TitleParams(
            text='Trends in Dam Construction - Long-term Structural Changes',
            fontSize=16,
            fontWeight=700,
            anchor='start',
            offset=20
        )
    ).configure_axis(
        labelFontSize=10,
        titleFontSize=11
    ).configure_header(
        labelFontSize=13,
        labelFontWeight=600,
        labelColor='#0C4A6E'
    ).configure_view(
        strokeWidth=0
    )
    
    return chart

In [296]:
# Define all dams with their coordinates and date ranges
dams_config = {
    'upper_yeywa': {
        'name': 'Upper Yeywa',
        'dam_coords': [97.102056, 22.242136],
        'res_coords': [97.133449, 22.249188],
        'start_date': '2014-01-01',
        'end_date': '2026-01-31'
    },
    'upper_kengtawng': {
        'name': 'Upper Kengtawng',
        'dam_coords': [98.184919, 20.745618],
        'res_coords': [98.182377, 20.747335],
        'start_date': '2014-01-01',
        'end_date': '2026-01-31'
    },
    'tha_htay': {
        'name': 'Tha Htay',
        'dam_coords': [94.380166, 18.641147],
        'res_coords': [94.377983, 18.640823],
        'start_date': '2018-01-01',
        'end_date': '2026-01-31'
    },
    'mone_chaung': {
        'name': 'Mone Chaung',
        'dam_coords': [94.253877, 20.478451],
        'res_coords': [94.256591, 20.480897],
        'start_date': '2018-01-01',
        'end_date': '2026-01-31'
    },
    'shwegyin': {
        'name': 'Shwegyin',
        'dam_coords': [96.932991, 17.968085],
        'res_coords': [96.931837, 17.970544],
        'start_date': '2014-01-01',
        'end_date': '2026-01-31'
    },
    'zawgyi_2': {
        'name': 'Zawgyi 2',
        'dam_coords': [96.87268274411485, 21.57494623482681],
        'res_coords': [96.874252, 21.575166],
        'start_date': '2014-01-01',
        'end_date': '2026-01-31'
    },
    'zawgyi_1': {
        'name': 'Zawgyi 1',
        'dam_coords': [96.896808, 21.398399],
        'res_coords': [96.897199, 21.398922],
        'start_date': '2014-01-01',
        'end_date': '2026-01-31'
    }
}

# Define buffer sizes to test (in meters)
buffer_sizes = [150, 200]

# Initialize lists to store all data
all_dam_data = []
all_reservoir_data = []

# Process each dam with each buffer size
for buffer_dam in buffer_sizes:
    for dam_id, config in dams_config.items():
        print(f"\n{'='*80}")
        print(f"Processing: {config['name']} (Buffer: {buffer_dam}m)")
        print(f"{'='*80}")
        
        try:
            # Extract dam data with specific buffer
            df = process_dam_data(
                config['dam_coords'], 
                config['res_coords'], 
                config['start_date'], 
                config['end_date'],
                buffer_dam=buffer_dam,
                buffer_res=40
            )
            
            # Process dam site
            df_dam = postprocess_dam_data(df, label='dam')
            df_dam = df_dam.reset_index()
            df_dam['dam_id'] = dam_id
            df_dam['dam_name'] = config['name']
            df_dam['buffer_dam'] = buffer_dam
            df_dam['buffer_label'] = f'{buffer_dam}m'
            all_dam_data.append(df_dam)
            
            # Process reservoir site
            df_res = postprocess_dam_data(df, label='reservoir')
            df_res = df_res.reset_index()
            df_res['dam_id'] = dam_id
            df_res['dam_name'] = config['name']
            df_res['buffer_dam'] = buffer_dam
            df_res['buffer_label'] = f'{buffer_dam}m'
            all_reservoir_data.append(df_res)
            
            print(f"✓ {config['name']} ({buffer_dam}m): Dam data collected ({len(df_dam)} records)")
            print(f"✓ {config['name']} ({buffer_dam}m): Reservoir data collected ({len(df_res)} records)")
            
        except Exception as e:
            print(f"✗ Error processing {config['name']} ({buffer_dam}m): {e}")

# Combine all data into single dataframes
combined_dam_df = pd.concat(all_dam_data, ignore_index=True)
combined_reservoir_df = pd.concat(all_reservoir_data, ignore_index=True)

# Reorder columns to have dam identifiers first
dam_columns = ['dam_id', 'dam_name', 'buffer_dam', 'buffer_label', 'date'] + [col for col in combined_dam_df.columns if col not in ['dam_id', 'dam_name', 'buffer_dam', 'buffer_label', 'date']]
combined_dam_df = combined_dam_df[dam_columns]
combined_reservoir_df = combined_reservoir_df[dam_columns]

# Save to CSV
combined_dam_df.to_csv('all_dams_backscatter_trends_multibuffer.csv', index=False)
combined_reservoir_df.to_csv('all_reservoirs_backscatter_trends_multibuffer.csv', index=False)

print(f"\n{'='*80}")
print(f"Processing Complete!")
print(f"Total dam records: {len(combined_dam_df)}")
print(f"Total reservoir records: {len(combined_reservoir_df)}")
print(f"{'='*80}")



Processing: Upper Yeywa (Buffer: 150m)
Total NaNs in Trend: 0
Total NaNs in Trend: 0
✓ Upper Yeywa (150m): Dam data collected (132 records)
✓ Upper Yeywa (150m): Reservoir data collected (132 records)

Processing: Upper Kengtawng (Buffer: 150m)
Total NaNs in Trend: 0
Total NaNs in Trend: 0
✓ Upper Kengtawng (150m): Dam data collected (135 records)
✓ Upper Kengtawng (150m): Reservoir data collected (135 records)

Processing: Tha Htay (Buffer: 150m)
Total NaNs in Trend: 0
Total NaNs in Trend: 0
✓ Tha Htay (150m): Dam data collected (97 records)
✓ Tha Htay (150m): Reservoir data collected (97 records)

Processing: Mone Chaung (Buffer: 150m)
Total NaNs in Trend: 0
Total NaNs in Trend: 0
✓ Mone Chaung (150m): Dam data collected (97 records)
✓ Mone Chaung (150m): Reservoir data collected (97 records)

Processing: Shwegyin (Buffer: 150m)
Total NaNs in Trend: 0
Total NaNs in Trend: 0
✓ Shwegyin (150m): Dam data collected (132 records)
✓ Shwegyin (150m): Reservoir data collected (132 records)


In [297]:

# First, load the combined data if not already loaded
combined_dam_df = pd.read_csv('all_dams_backscatter_trends_multibuffer.csv')
combined_dam_df['date'] = pd.to_datetime(combined_dam_df['date'])

# # Calculate residuals if not already present
# if 'residual' not in combined_dam_df.columns:
#     combined_dam_df['residual'] = combined_dam_df['VV_filled'] - combined_dam_df['trend'] - combined_dam_df['seasonal']

# # Plot all dams (or first 6 if more than 6 exist) - showing only 150m buffer for grid
# chart = plot_trends_grid(combined_dam_df[combined_dam_df['buffer_label'] == '150m'])
# chart



In [298]:
import warnings
warnings.filterwarnings('ignore', message='Automatically deduplicated selection parameter')

# Interactive dashboard: Select a dam and buffer size, view all three components side by side
def plot_dam_dashboard(combined_df):
    """
    Create an interactive dashboard with dropdowns to select a dam and buffer size, 
    then view trend, seasonal, and residual components side by side.
    Y-axes remain constant for each dam across buffer sizes for easier comparison.
    
    Args:
        combined_df: DataFrame with all dam data including trend, seasonal, residual, 
                     and buffer columns
    
    Returns:
        Altair chart object with interactive dropdowns
    """
    # Ensure residual column exists
    if 'residual' not in combined_df.columns:
        combined_df['residual'] = combined_df['VV_filled'] - combined_df['trend'] - combined_df['seasonal']
    
    # Prepare data
    plot_data = combined_df.copy()
    plot_data['date'] = pd.to_datetime(plot_data['date'])
    
    # Get unique dams and buffers for dropdowns
    dam_names = sorted(plot_data['dam_name'].unique())
    buffer_labels = sorted(plot_data['buffer_label'].unique()) if 'buffer_label' in plot_data.columns else ['150m']
    
    # Create dam dropdown selection
    dam_dropdown = alt.binding_select(options=dam_names, name='Select Dam: ')
    dam_selection = alt.selection_point(
        name='dam_select',
        fields=['dam_name'],
        bind=dam_dropdown,
        value=dam_names[0]
    )
    
    # Create buffer dropdown selection
    buffer_dropdown = alt.binding_select(options=buffer_labels, name='Buffer Size: ')
    buffer_selection = alt.selection_point(
        name='buffer_select',
        fields=['buffer_label'],
        bind=buffer_dropdown,
        value=buffer_labels[1] if len(buffer_labels) > 1 else buffer_labels[0]
    )
    
    # Base properties with both selections
    base = alt.Chart(plot_data).add_params(
        dam_selection,
        buffer_selection
    ).transform_filter(
        dam_selection
    ).transform_filter(
        buffer_selection
    )
    
    # Y-axes: trend auto-scales per dam (different for each dam, same between buffers)
    # Seasonal uses a fixed domain across all dams
    seasonal_domain = [plot_data['seasonal'].min() * 1.1, plot_data['seasonal'].max() * 1.1]
    
    # Chart 1: Trend Component (Blue Line) - auto-scaling y-axis per dam
    trend_chart = base.mark_line(
        strokeWidth=3,
        color='#0C4A6E'
    ).encode(
        x=alt.X('date:T', 
                title='Date',
                axis=alt.Axis(format='%Y', labelAngle=0)),
        y=alt.Y('trend:Q', 
                title='Backscatter (dB)',
                scale=alt.Scale(zero=False)),
        tooltip=[
            alt.Tooltip('dam_name:N', title='Dam'),
            alt.Tooltip('buffer_label:N', title='Buffer'),
            alt.Tooltip('date:T', title='Date', format='%Y-%m-%d'),
            alt.Tooltip('trend:Q', title='Trend (dB)', format='.2f')
        ]
    ).properties(
        width=350,
        height=300,
        title=alt.TitleParams(
            text='Trend - Long-term Structural Changes',
            fontSize=13,
            fontWeight=600,
            color='#0C4A6E',
            anchor='start'
        )
    )
    
    # Chart 2: Seasonal Component (Green Line)
    seasonal_chart = base.mark_line(
        strokeWidth=3,
        color='#16A34A'
    ).encode(
        x=alt.X('date:T', 
                title='Date',
                axis=alt.Axis(format='%Y', labelAngle=0)),
        y=alt.Y('seasonal:Q', 
                title='Backscatter (dB)',
                scale=alt.Scale(domain=seasonal_domain)),
        tooltip=[
            alt.Tooltip('dam_name:N', title='Dam'),
            alt.Tooltip('buffer_label:N', title='Buffer'),
            alt.Tooltip('date:T', title='Date', format='%Y-%m-%d'),
            alt.Tooltip('seasonal:Q', title='Seasonal (dB)', format='.2f')
        ]
    ).properties(
        width=350,
        height=300,
        title=alt.TitleParams(
            text='Seasonal - Monsoon Cycles',
            fontSize=13,
            fontWeight=600,
            color='#16A34A',
            anchor='start'
        )
    )
    
    # Chart 3: Residuals (Dark Gray Scatter) - Auto-scales per dam for better anomaly visibility
    residual_chart = base.mark_circle(
        size=30,
        color='#1F2937',
        opacity=0.7
    ).encode(
        x=alt.X('date:T', 
                title='Date',
                axis=alt.Axis(format='%Y', labelAngle=0)),
        y=alt.Y('residual:Q', 
                title='Backscatter (dB)',
                scale=alt.Scale(zero=False)),
        tooltip=[
            alt.Tooltip('dam_name:N', title='Dam'),
            alt.Tooltip('buffer_label:N', title='Buffer'),
            alt.Tooltip('date:T', title='Date', format='%Y-%m-%d'),
            alt.Tooltip('residual:Q', title='Residual (dB)', format='.2f')
        ]
    ).properties(
        width=350,
        height=300,
        title=alt.TitleParams(
            text='Residuals - Unexplained Anomalies',
            fontSize=13,
            fontWeight=600,
            color='#1F2937',
            anchor='start'
        )
    )
    
    # Combine charts horizontally
    dashboard = (trend_chart | seasonal_chart | residual_chart).configure_view(
        strokeWidth=0
    ).configure_axis(
        labelFontSize=11,
        titleFontSize=12
    ).properties(
        title=alt.TitleParams(
            text='Dam Construction Time Series Analysis - STL Decomposition Components',
            subtitle='Buffer size refers to the radius around the point location of dam used in the analysis.',
            fontSize=16,
            fontWeight=700,
            anchor='start',
            offset=20,
            subtitleFontSize=12,
            subtitleColor='#6B7280'
        )
    )
    
    return dashboard

# Create and display the interactive dashboard with dam and buffer filters
dashboard = plot_dam_dashboard(combined_dam_df)
dashboard

alt.HConcatChart(...)

#### Upper Yeywa Hydropower Project

<table>
<tr><td align="center"><b>2017</b></td><td align="center"><b>2019</b></td><td align="center"><b>2023</b></td><td align="center"><b>2025</b></td></tr>
<tr>
<td><img src="/myanmar-economic-monitor/upper_yeywa_2017_SE.png" width="240"></td>
<td><img src="/myanmar-economic-monitor/upper_yeywa_2019_GE.png" width="240"></td>
<td><img src="/myanmar-economic-monitor/upper_yeywa_2023_GE.png" width="240"></td>
<td><img src="/myanmar-economic-monitor/upper_yeywa_2025_SE.png" width="240"></td>
</tr>
</table>

#### Upper Kengtawng Hydroelectric Plant

<table>
<tr><td align="center"><b>2015</b></td><td align="center"><b>2017</b></td><td align="center"><b>2021</b></td><td align="center"><b>2025</b></td></tr>
<tr>
<td><img src="/myanmar-economic-monitor/upper_kengtawng_composite_2015_buff1000.png" width="240"></td>
<td><img src="/myanmar-economic-monitor/upper_kengtawng_composite_2017_buff1000.png" width="240"></td>
<td><img src="/myanmar-economic-monitor/upper_kengtawng_composite_2021_buff1000.png" width="240"></td>
<td><img src="/myanmar-economic-monitor/upper_kengtawng_composite_2025_buff1000.png" width="240"></td>
</tr>
</table>

<h5> Construction Site Dynamics (Structural Growth vs. Volatility) </h5>

The Upper Kengtawng analysis demonstrates a successful construction cycle that, while delayed from its original 2021 target, has avoided the sudden conflict-driven shocks as experienced by some other projects. The project, located on the Nam Teng River in Southern Shan State, is a rockfill dam with an installed capacity of 51 MW. The trend component above captures the transition from foundational earthworks to the completion of the main dam structure. At the tail end there's a gradual downward slope. This likely reflects the transition from "construction" to "containment." As the dam nears 100% completion [(reported at 90%+ in late 2025)](https://www.myanmaritv.com/news/development-works-shan-state-chief-minister-makes-inspection-tour), the area behind the dam begins to fill, and the presence of water (a specular reflector) starts to pull the overall pixel intensity down.

<h5> Upstream Inundation (Reservoir Signature) </h5>

The second plot shows that the upstream control point indicates a definitive land-cover transition consistent with reservoir inundation. The signal exhibits a precipitous decline from a baseline of -9.0 dB to approximately -15.0 dB during the 2018 calendar year, marking the initial flooding event.The continued stability of the signal through 2026 confirms that the reservoir footprint has remained consistent, even as the project reached near completion as of late 2025.

<h5> Construction Site Dynamics (Structural Growth vs. Volatility) </h5>

Tha Htay reached ~80% completion before the major [conflict-driven shutdown](https://nssmy.com/news_detail/17542/1/1). The main dam (a massive 90-meter high embankment) and the associated power station infrastructure are already physically present. Even after the halt, the upward trend in the analysis reflects the accumulated physical volume of a nearly finished dam rather than ongoing construction. The project reached a high state of structural completion (approx. 80%) before the April 2024 halt; therefore, the radar is capturing the high reflectance of the massive embankment and spillway. The stabilization of the residuals post-2024 is the true indicator of the construction halt, signaling the absence of mobile machinery and operational logistical movement.

<h5> Upstream Inundation (Reservoir Signature) </h5>

The upstream analysis shows that despite the reported construction halt in April 2024, the Tha Htay reservoir is functionally active. The permanent shift in backscatter that began in 2021 suggests the dam structure is high enough to maintain a consistent upstream pool. The current stability of the signal through April 2026 confirms that while the "human" work on the power plant is stalled, the environmental transformation of the Tha Htay Creek into a 111 reservoir is constantly happening. The radar data captured an unexplained anomaly in residuals in early 2025. This likely represents a significant flash flood or debris flow, a common issue in the region, which has suffered repeated site damage from heavy rains in recent years. 

The signal trajectory is broadly consistent with what is known about Mone Chaung: a long-operational dam with no active new construction since 2004, showing a stable middle period followed by a structurally declining trend that correlates well with post-coup deterioration of Myanmar's hydropower sector. The sharp structural decline beginning in late 2022 and accelerating through 2023–2026 (reaching ~−6.8 dB by early 2026) is the most consequential feature. Myanmar's existing hydropower plants have faced increasing disruptions due to conflict and infrastructure damage following the 2021 coup Seasia, and this declining backscatter is consistent with reduced operational intensity - less mechanical activity, equipment removal, or simply higher persistent water levels with less drawdown-driven roughness. 

The SAR-derived signal for Shwegyin is broadly consistent with the facility's known operational history. Construction began in 2003 and the plant entered commercial operation in 2011, meaning the time series from 2014 onward reflects post-commissioning behavior rather than active construction. The declining backscatter trend toward 2016 is consistent with reservoir maturation and stabilization following initial impoundment. 


The post-2022 upward trend and elevated residual variance coincide with a deteriorating security environment in Shwegyin Township, Bago Region. Documented conflict activity in the area, including armed clashes between junta forces and Karen National Union fighters, displacement of civilian populations, and artillery and airstrike activity, points to conditions that would plausibly disrupt normal plant operations and maintenance. For a 75 MW facility operating in an area of active conflict, reduced operational activity, deferred maintenance, and personnel disruption are all factors that could contribute to the observed changes in surface backscatter character.